In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!unzip "/content/drive/MyDrive/dataset/animal_dataset.zip" -d "/content/dataset"


Streaming output truncated to the last 5000 lines.
  inflating: /content/dataset/cheetah-resize-224/resize-224/00000236_224resized.png  
  inflating: /content/dataset/cheetah-resize-224/resize-224/00000239_224resized.png  
  inflating: /content/dataset/cheetah-resize-224/resize-224/00000240_224resized.png  
  inflating: /content/dataset/cheetah-resize-224/resize-224/00000242_224resized.png  
  inflating: /content/dataset/cheetah-resize-224/resize-224/00000244_224resized.png  
  inflating: /content/dataset/cheetah-resize-224/resize-224/00000245_224resized.png  
  inflating: /content/dataset/cheetah-resize-224/resize-224/00000246_224resized.png  
  inflating: /content/dataset/cheetah-resize-224/resize-224/00000247_224resized.png  
  inflating: /content/dataset/cheetah-resize-224/resize-224/00000248_224resized.png  
  inflating: /content/dataset/cheetah-resize-224/resize-224/00000250_224resized.png  
  inflating: /content/dataset/cheetah-resize-224/resize-224/00000251_224resized.png  
  i

In [ ]:
!ls /content


dataset  drive	sample_data


In [ ]:
!ls /content/dataset


cheetah-resize-224  fox-resize-512    lion-resize-300	wolf-resize-224
cheetah-resize-300  hyena-resize-224  lion-resize-512	wolf-resize-300
cheetah-resize-512  hyena-resize-300  tiger-resize-224	wolf-resize-512
fox-resize-224	    hyena-resize-512  tiger-resize-300
fox-resize-300	    lion-resize-224   tiger-resize-512


In [ ]:
DATASET_PATH = "/content/animal_dataset"



In [ ]:
!find /content -type d -maxdepth 3


find: warning: you have specified the global option -maxdepth after the argument -type, but global options are not positional, i.e., -maxdepth affects tests specified before it as well as those specified after it.  Please specify global options before other arguments.
/content
/content/.config
/content/.config/configurations
/content/.config/logs
/content/.config/logs/2025.11.20
/content/dataset
/content/dataset/lion-resize-224
/content/dataset/lion-resize-224/lion-resize-224
/content/dataset/wolf-resize-300
/content/dataset/wolf-resize-300/wolf-resize-300
/content/dataset/hyena-resize-300
/content/dataset/hyena-resize-300/resize-300
/content/dataset/tiger-resize-512
/content/dataset/tiger-resize-512/tiger-resize-512
/content/dataset/cheetah-resize-224
/content/dataset/cheetah-resize-224/resize-224
/content/dataset/lion-resize-300
/content/dataset/lion-resize-300/lion-resize-300
/content/dataset/lion-resize-512
/content/dataset/lion-resize-512/lion-resize-512
/content/dataset/fox-resiz

In [ ]:
!ls /content


sample_data


In [ ]:
import os
import cv2
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
DATASET_PATH = "/content/dataset"
IMG_SIZE = 128

In [ ]:
IMG_SIZE = 128
X = []
y = []

DATASET_PATH = "/content/dataset"

# Import HOG function
from skimage.feature import hog

# Loop through all folders and subfolders
for root, dirs, files in os.walk(DATASET_PATH):

    for file in files:
        # Only accept PNG and JPG images
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            img_path = os.path.join(root, file)

            # Parent folder name = class name
            class_name = os.path.basename(os.path.dirname(img_path))

            # Assign unique numeric label for each class name
            label = hash(class_name) % 10000

            try:
                # Read image
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"Warning: Could not read image at {img_path}. Skipping.")
                    continue
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

                # Extract HOG features
                features = hog(
                    img,
                    orientations=9,
                    pixels_per_cell=(8, 8),
                    cells_per_block=(2, 2),
                    block_norm='L2-Hys'
                )

                X.append(features)
                y.append(label)

            except Exception as e:
                print("Error:", e)
                pass

X = np.array(X)
y = np.array(y)

print("Total images loaded:", len(X))
print("Feature vector size:", X.shape)
print("Unique classes:", len(np.unique(y)))

Total images loaded: 5169
Feature vector size: (5169, 8100)
Unique classes: 15


In [ ]:
model = DecisionTreeClassifier(max_depth=40, random_state=42)
model.fit(X_train, y_train)

print("Training Completed!")

Training Completed!


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [ ]:
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10,7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

print(classification_report(y_test, y_pred))

NameError: name 'y_pred' is not defined